[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Daniel-534/Astroestadistica/blob/main/Clases/AstroestadisticaBayesianStatisticsGW.ipynb)

In [ ]:
# Run this cell once to install the required packages
# (already available on most GW-focused Conda/Docker environments)
import subprocess, sys
pkgs = ["bilby", "bilby_pipe", "dynesty", "lalsuite", "python-ligo-lw",
        "corner", "matplotlib", "numpy", "scipy", "astropy"]
subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet"] + pkgs)
print('Installation complete.')

---
<a id='sec1'></a>
## 1 · Bayesian inference fundamentals

### 1.1 Bayes theorem

Let $\boldsymbol{\theta}$ be a vector of unknown parameters and $d$ the observed data.
Bayes theorem reads

$$
\boxed{p(\boldsymbol{\theta}\mid d) = \frac{p(d\mid\boldsymbol{\theta})\,p(\boldsymbol{\theta})}{p(d)}}
$$

| Symbol | Name | Role |
|---|---|---|
| $p(\boldsymbol{\theta}\mid d)$ | **Posterior** | Probability of parameters given data — what we want |
| $p(d\mid\boldsymbol{\theta})$ | **Likelihood** | Probability of observing $d$ if $\boldsymbol{\theta}$ is true |
| $p(\boldsymbol{\theta})$ | **Prior** | Probability of $\boldsymbol{\theta}$ before seeing any data |
| $p(d)$ | **Evidence** (marginal likelihood) | Normalisation constant; key for model comparison |

The evidence is

$$
p(d) = \int p(d\mid\boldsymbol{\theta})\,p(\boldsymbol{\theta})\,d\boldsymbol{\theta}.
$$

### 1.2 The update picture

Bayesian inference is a **coherent rule for updating beliefs**.  The prior encodes
what we know *before* the observation; the likelihood tells us how probable the
data are for each candidate parameter value; the posterior synthesises both.

### 1.3 The posterior is a distribution, not a point

Unlike maximum-likelihood estimation (which gives a single best-fit value), Bayesian
inference delivers a **full probability distribution** over $\boldsymbol{\theta}$.  This
allows us to

* quantify **credible intervals** (e.g. 90 % CI),
* **marginalise** over nuisance parameters,
* perform principled **model comparison** via the evidence ratio (Bayes factor).

### 1.4 Bayes factors and model selection

Given two hypotheses $\mathcal{H}_1$ and $\mathcal{H}_2$, the Bayes factor is

$$
\mathcal{B}_{12} = \frac{p(d\mid\mathcal{H}_1)}{p(d\mid\mathcal{H}_2)}.
$$

In GW astronomy this is used to distinguish **signal vs noise** hypotheses and to
compare source models (e.g. precessing vs aligned-spin).


In [ ]:
"""Toy 1-D Bayesian update: estimating a distance.

We have a Gaussian prior on luminosity distance d_L ~ N(300, 100) Mpc
and a Gaussian likelihood from a single GW measurement.
We visualise how the posterior sharpens as we 'add' more measurements.
"""

import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm

# ---- prior parameters ----
mu_prior, sigma_prior = 300.0, 100.0  # Mpc

# ---- simulated measurements (GW-like): true d_L = 250 Mpc
true_dL = 250.0
meas_sigma = 60.0  # measurement noise per event
rng = np.random.default_rng(42)
measurements = rng.normal(true_dL, meas_sigma, size=5)

d_grid = np.linspace(0, 700, 1000)

fig, axes = plt.subplots(1, 3, figsize=(14, 4), sharey=False)
colors = plt.cm.plasma(np.linspace(0.15, 0.85, 5))

n_events_list = [0, 1, 5]
for ax, n in zip(axes, n_events_list):
    # Start from prior
    log_posterior = norm.logpdf(d_grid, mu_prior, sigma_prior)
    # Multiply in likelihoods one by one
    for i in range(n):
        log_posterior += norm.logpdf(d_grid, measurements[i], meas_sigma)
    posterior = np.exp(log_posterior - log_posterior.max())
    posterior /= np.trapezoid(posterior, d_grid)

    ax.plot(d_grid, norm.pdf(d_grid, mu_prior, sigma_prior),
            'k--', lw=1.5, label='Prior', alpha=0.6)
    ax.fill_between(d_grid, posterior, alpha=0.4, color='steelblue')
    ax.plot(d_grid, posterior, color='steelblue', lw=2,
            label=f'Posterior ({n} event{"s" if n!=1 else ""})')
    ax.axvline(true_dL, color='crimson', ls=':', lw=1.8, label=f'True $d_L$={true_dL} Mpc')
    ax.set_xlabel(r'$d_L$ [Mpc]', fontsize=12)
    ax.set_ylabel('Probability density', fontsize=11)
    ax.legend(fontsize=9)
    ax.set_title(f'{n} GW event{"s" if n!=1 else ""} observed', fontsize=11)

fig.suptitle('Bayesian update of $d_L$ posterior', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()
print('As more events are observed the posterior narrows around the true value.')

---
<a id='sec2'></a>
## 2 · Bayesian inference in GW astronomy

### 2.1 The parameter space

A CBC signal is characterised by $\sim$15 parameters:

| Category | Parameters |
|---|---|
| **Masses** | $m_1,m_2$ or equivalently chirp mass $\mathcal{M}$, mass ratio $q$ |
| **Spins** | $\chi_{1x},\chi_{1y},\chi_{1z},\chi_{2x},\chi_{2y},\chi_{2z}$ |
| **Extrinsic** | $d_L$, right ascension $\alpha$, declination $\delta$, inclination $\iota$, polarisation $\psi$, coalescence time $t_c$, phase $\phi_c$ |

The intrinsic parameters determine the **waveform shape**; the extrinsic parameters
set the **amplitude and phase at the detectors**.

### 2.2 The chirp mass

The single parameter that controls the leading-order GW frequency evolution is the
**chirp mass**:

$$
\mathcal{M} = \frac{(m_1 m_2)^{3/5}}{(m_1+m_2)^{1/5}}
$$

It is measured with exceptional accuracy from the rate of frequency change
$\dot{f}\propto\mathcal{M}^{5/3}f^{11/3}$.

### 2.3 Noise model and the inner product

Detector output: $d(t)=h(t;\boldsymbol{\theta})+n(t)$, where $n(t)$ is stationary
Gaussian noise with one-sided PSD $S_n(f)$.

The **noise-weighted inner product** between two real time series $a(t)$ and $b(t)$ is

$$
\langle a | b \rangle \;=\; 4\,\Re\int_0^\infty \frac{\tilde{a}(f)\,\tilde{b}^*(f)}{S_n(f)}\,df
$$

where $\tilde{a}(f)$ is the Fourier transform of $a(t)$.

### 2.4 The GW log-likelihood

Under a Gaussian noise assumption the log-likelihood for a single detector is

$$
\ln p(d\mid\boldsymbol{\theta}) = -\frac{1}{2}\langle d - h(\boldsymbol{\theta})\,|\,d - h(\boldsymbol{\theta})\rangle
+ \text{const}
$$

Expanding:

$$
\ln p(d\mid\boldsymbol{\theta}) = \langle d | h(\boldsymbol{\theta})\rangle
                                  - \frac{1}{2}\langle h(\boldsymbol{\theta}) | h(\boldsymbol{\theta})\rangle
                                  + \text{const}
$$

The matched-filter **signal-to-noise ratio** (SNR) for an optimal filter $h$ is

$$
\rho = \frac{\langle d | h \rangle}{\sqrt{\langle h | h \rangle}}
$$

For a **network** of detectors, log-likelihoods from individual detectors are
simply added (assuming independent noise):

$$
\ln p(d_{\rm net}\mid\boldsymbol{\theta}) = \sum_{k}\ln p(d_k\mid\boldsymbol{\theta})
$$

### 2.5 Dependence of amplitude on $d_L$ and $\iota$

The two GW polarisations scale as:

$$
h_+ \propto \frac{1+\cos^2\iota}{2}\,\frac{1}{d_L}, \qquad
h_\times \propto \cos\iota\,\frac{1}{d_L}
$$

This creates the famous **$d_L$–$\iota$ degeneracy**: a face-on binary
($\iota=0$) at large distance can produce the same amplitude as an edge-on binary
($\iota=\pi/2$) at smaller distance.  A two-detector network partially breaks this
degeneracy through polarisation information; a third detector does so further.

### 2.6 Sky localisation

Sky position enters through the **detector response functions** (antenna patterns)
$F_+^{(k)}(\alpha,\delta,\psi)$ and $F_\times^{(k)}(\alpha,\delta,\psi)$.  The
time-of-arrival difference between two detectors separated by baseline $L$ gives
a directional constraint:

$$
\Delta t = \frac{L\cos\theta}{c}
$$

where $\theta$ is the angle between the source direction and the baseline.  For
LIGO Hanford–Livingston ($L=3000$ km), $\Delta t_{\rm max}\approx10$ ms.


In [ ]:
"""Visualise LIGO-Hanford antenna patterns F+ and Fx
as a function of sky position for a fixed polarisation angle.
"""

import numpy as np
import matplotlib.pyplot as plt

def ligo_hanford_antenna(ra, dec, psi=0.0, gmst=0.0):
    """Simplified LIGO-Hanford antenna patterns (equatorial coordinates).
    Reference: Jaranowski & Krolak (2009), Chapter 4.
    """
    # LIGO-Hanford detector tensor (approximate, in geographic frame)
    # Arm azimuths: ~N35.9°W and ~S54.1°W
    phi_det = np.radians(-119.408)  # longitude
    lam_det = np.radians(46.455)    # latitude
    # Source direction in detector frame
    ha = gmst - ra  # hour angle
    sinlat, coslat = np.sin(lam_det), np.cos(lam_det)
    sindec, cosdec = np.sin(dec), np.cos(dec)
    sinha,  cosha  = np.sin(ha),  np.cos(ha)
    # Convenient combinations
    a1 = 0.5*(1 + sinlat**2)*cosdec**2*np.cos(2*ha)
    a2 = sinlat*sindec*cosdec*np.cos(ha)
    a3 = 0.25*(1 + sinlat**2)*(1 + sindec**2)
    b1 = sinlat*cosdec**2*np.sin(2*ha)
    b2 = coslat*sindec
    b3 = 0.5*sinlat*(1 + sindec**2)*np.sin(2*ha)
    # Very rough estimates (illustrative)
    Fp = np.sin(2*ha)*cosdec**2*np.cos(2*psi) - 2*np.cos(ha)*sindec*cosdec*np.sin(2*psi)
    Fc = np.sin(2*ha)*cosdec**2*np.sin(2*psi) + 2*np.cos(ha)*sindec*cosdec*np.cos(2*psi)
    return Fp, Fc

# Sky grid
ra_grid  = np.linspace(0, 2*np.pi, 360)
dec_grid = np.linspace(-np.pi/2, np.pi/2, 180)
RA, DEC = np.meshgrid(ra_grid, dec_grid)

Fp, Fc = ligo_hanford_antenna(RA, DEC, psi=0.0, gmst=np.pi/3)
sensitivity = np.sqrt(Fp**2 + Fc**2)

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
titles   = [r'$F_+$ (plus)', r'$F_\times$ (cross)', r'$\sqrt{F_+^2+F_\times^2}$ (total sensitivity)']
datasets = [Fp, Fc, sensitivity]
cmaps    = ['RdBu_r', 'RdBu_r', 'viridis']

for ax, data, title, cmap in zip(axes, datasets, titles, cmaps):
    im = ax.pcolormesh(np.degrees(ra_grid), np.degrees(dec_grid), data,
                       cmap=cmap, shading='auto')
    plt.colorbar(im, ax=ax)
    ax.set_title(title, fontsize=11)
    ax.set_xlabel('Right Ascension [deg]', fontsize=10)
    ax.set_ylabel('Declination [deg]', fontsize=10)

fig.suptitle('LIGO-Hanford Antenna Patterns (illustrative, $\\psi=0$)', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()